In [2]:
from dotenv import load_dotenv
load_dotenv("/mnt/nfs/llmhub/MetaLm/sockett/genomeer/.env", override=True)

import os
print("LANGCHAIN_API_KEY:", os.environ.get("LANGCHAIN_API_KEY", "")[:15] + "...")
print("LANGCHAIN_PROJECT:", os.environ.get("LANGCHAIN_PROJECT"))


LANGCHAIN_API_KEY: lsv2_pt_ada18a3...
LANGCHAIN_PROJECT: genomeer-benchmark


In [ ]:
import sys, os, time, subprocess
sys.path.insert(0, "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src")

from langchain_openai import ChatOpenAI
from genomeer.agent.v2.BioAgent import BioAgent

# ─── Cerebras API keys (https://cloud.cerebras.ai) ────────────────────────────
CEREBRAS_KEY_1 =   # clé principale (main LLM)
CEREBRAS_KEY_2 =  # clé secondaire (bio_hint LLM)

LOCAL_TEST_DIR = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests"

# Cerebras (LPU ~2000 tok/s) — model id SANS préfixe "openai/"
MODEL    = "kimi-2b"
BASE_URL = "https://api.cerebras.ai/v1"

# Couper le raisonnement interne (économise 3000-8000 tokens par appel)
REASONING_OFF = {"reasoning_effort": "low"}

bio_hint_llm = ChatOpenAI(
    model=MODEL,
    base_url=BASE_URL,
    api_key=CEREBRAS_KEY_2,
    temperature=0.3,
    max_tokens=300,
    max_retries=3,
    extra_body=REASONING_OFF,
)

agent = BioAgent(
    path=LOCAL_TEST_DIR,
    llm=MODEL,
    source="Custom",
    base_url=BASE_URL,
    api_key=CEREBRAS_KEY_1,
    bio_hint_llm=None,
    use_tool_retriever=True,
    auto_start_artifacts=True,
    timeout_seconds=300,
)

# Patch le LLM principal créé par BioAgent pour appliquer le no-reasoning
if hasattr(agent, "llm"):
    agent.llm.extra_body = REASONING_OFF
    agent.llm.max_retries = 3
    print(f"Provider : Cerebras  |  Modele : {MODEL}  |  Reasoning: LOW")

# ─── Lancement de l'UI web (port 8911) en sous-process isolé ──────────────────
# Subprocess avec cwd=UI_DIR → cwd du kernel notebook intact → BioAgent's
# ./agent_debug.log reste dans /mnt/nfs/llmhub/MetaLm/ et non dans le dossier UI.
UI_DIR = "/mnt/nfs/llmhub/MetaLm/sockett/agent-ui"
_ui_env = os.environ.copy()
_ui_env["AGENT_COPILOT_SECRET"] = "test-local-secret-32chars-minimum!"
_ui_env["ALLOWED_ORIGIN"]       = "*"
# Inject genomeer src into subprocess PYTHONPATH (sys.path n'est pas hérité)
_ui_env["PYTHONPATH"] = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src" + os.pathsep + _ui_env.get("PYTHONPATH", "")

# Tuer tout ancien processus UI sur le port 8911 avant d'en relancer un
try:
    subprocess.run(["pkill", "-f", "uvicorn app.main:app.*8911"], timeout=5)
    time.sleep(0.5)
except Exception:
    pass

_ui_log = open(os.path.join(UI_DIR, "ui_uvicorn.log"), "ab")
ui_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8911", "--log-level", "warning"],
    cwd=UI_DIR,
    env=_ui_env,
    stdout=_ui_log,
    stderr=_ui_log,
)

# Attendre que l'UI réponde
import requests as _req
for _i in range(25):
    try:
        if _req.get("http://localhost:8911/", timeout=2).status_code in (200, 401, 404):
            print(f"UI Web   : http://<IP-serveur>:8911   (PID subprocess: {ui_proc.pid})")
            break
    except Exception:
        time.sleep(1)
else:
    print(f"UI Web   : ⚠️  pas démarrée — voir {UI_DIR}/ui_uvicorn.log")

print(f"Artifacts: http://127.0.0.1:8910/docs")
print(f"Kernel CWD: {os.getcwd()}   (doit rester /mnt/nfs/llmhub/MetaLm)")


/mnt/nfs/llmhub/sft_env/lib/python3.10/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



BioAgent_v3 CONFIGURATION
EFFECTIVE CONFIG :
  Path: /mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests
  Run Dir: .genomeer_runs
  Timeout Seconds: 300
  Llm: gpt-oss-120b
  Temperature: 0.7
  Use Tool Retriever: True
  Source: Custom
  Base Url: https://api.cerebras.ai/v1
  Api Key: ********3e5x

LangSmith tracing: ON (project=genomeer-benchmark)
Provider : Cerebras  |  Modele : gpt-oss-120b  |  Reasoning: LOW


INFO:     Started server process [438638]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


Artifacts service on http://127.0.0.1:8910/docs
UI Web   : http://<IP-serveur>:8911   (PID subprocess: 439074)
Artifacts: http://127.0.0.1:8910/docs
Kernel CWD: /mnt/nfs/llmhub/MetaLm   (doit rester /mnt/nfs/llmhub/MetaLm)


In [ ]:
import sys, os, time, subprocess
sys.path.insert(0, "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src")

from langchain_openai import ChatOpenAI
from genomeer.agent.v2.BioAgent import BioAgent

# ─── DeepSeek API keys (https://platform.deepseek.com/api_keys) ───────────────
# Plateforme officielle DeepSeek (pas OpenRouter). Modèle PAYANT.
DEEPSEEK_KEY_1 =""  # clé principale (main LLM)
DEEPSEEK_KEY_2 =""  # clé secondaire (bio_hint LLM)

LOCAL_TEST_DIR = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests"

# DeepSeek V4 Pro — depuis la plateforme officielle DeepSeek
MODEL    = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com/v1"

bio_hint_llm = ChatOpenAI(
    model=MODEL,
    base_url=BASE_URL,
    api_key=DEEPSEEK_KEY_2,
    temperature=0.3,
    max_tokens=300,
    max_retries=3,
)

agent = BioAgent(
    path=LOCAL_TEST_DIR,
    llm=MODEL,
    source="Custom",
    base_url=BASE_URL,
    api_key=DEEPSEEK_KEY_1,
    bio_hint_llm=None,
    use_tool_retriever=True,
    auto_start_artifacts=True,
    timeout_seconds=300,
)

if hasattr(agent, "llm"):
    agent.llm.max_retries = 3
    print(f"Provider : DeepSeek (officiel)  |  Modele : {MODEL}")

# ─── Lancement de l'UI web (port 8911) en sous-process isolé ──────────────────
UI_DIR = "/mnt/nfs/llmhub/MetaLm/sockett/agent-ui"
_ui_env = os.environ.copy()
_ui_env["AGENT_COPILOT_SECRET"] = "test-local-secret-32chars-minimum!"
_ui_env["ALLOWED_ORIGIN"]       = "*"
_ui_env["PYTHONPATH"] = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src" + os.pathsep + _ui_env.get("PYTHONPATH", "")

try:
    subprocess.run(["pkill", "-f", "uvicorn app.main:app.*8911"], timeout=5)
    time.sleep(0.5)
except Exception:
    pass

_ui_log = open(os.path.join(UI_DIR, "ui_uvicorn.log"), "ab")
ui_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8911", "--log-level", "warning"],
    cwd=UI_DIR,
    env=_ui_env,
    stdout=_ui_log,
    stderr=_ui_log,
)

import requests as _req
for _i in range(25):
    try:
        if _req.get("http://localhost:8911/", timeout=2).status_code in (200, 401, 404):
            print(f"UI Web   : http://<IP-serveur>:8911   (PID subprocess: {ui_proc.pid})")
            break
    except Exception:
        time.sleep(1)
else:
    print(f"UI Web   : ⚠️  pas démarrée — voir {UI_DIR}/ui_uvicorn.log")

print(f"Artifacts: http://127.0.0.1:8910/docs")
print(f"Kernel CWD: {os.getcwd()}   (doit rester /mnt/nfs/llmhub/MetaLm)")


/mnt/nfs/llmhub/sft_env/lib/python3.10/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



BioAgent_v3 CONFIGURATION
EFFECTIVE CONFIG :
  Path: /mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests
  Run Dir: .genomeer_runs
  Timeout Seconds: 300
  Llm: deepseek-v4-pro
  Temperature: 0.7
  Use Tool Retriever: True
  Source: Custom
  Base Url: https://api.deepseek.com/v1
  Api Key: ********4ae5

LangSmith tracing: ON (project=genomeer-benchmark)
Provider : DeepSeek (officiel)  |  Modele : deepseek-v4-pro


INFO:     Started server process [1297297]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


Artifacts service on http://127.0.0.1:8910/docs
UI Web   : http://<IP-serveur>:8911   (PID subprocess: 1297935)
Artifacts: http://127.0.0.1:8910/docs
Kernel CWD: /mnt/nfs/llmhub/MetaLm   (doit rester /mnt/nfs/llmhub/MetaLm)


INFO:     127.0.0.1:49634 - "POST /api/v1/artifacts/runs/create/31 HTTP/1.1" 200 OK
INFO:     127.0.0.1:49640 - "POST /api/v1/artifacts/runs/31/upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:39804 - "POST /api/v1/artifacts/runs/create/56 HTTP/1.1" 200 OK
INFO:     127.0.0.1:39808 - "POST /api/v1/artifacts/runs/56/upload HTTP/1.1" 200 OK


In [ ]:
import sys, os, time, subprocess
sys.path.insert(0, "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src")

from langchain_openai import ChatOpenAI
from genomeer.agent.v2.BioAgent import BioAgent

# ─── Main LLM : DeepSeek V4 Pro (officiel) ────────────────────────────────────
DEEPSEEK_KEY  = ""
MAIN_MODEL    = "deepseek-v4-pro"
MAIN_BASE_URL = "https://api.deepseek.com/v1"

# ─── bio_hint LLM : Apertus-8B-BioInstruct (fine-tuned local) ─────────────────
APERTUS_MODEL    = "Apertus-8B-BioInstruct"
APERTUS_BASE_URL = "http://127.0.0.1:8001/v1"

def _apertus_alive():
    import urllib.request
    try:
        urllib.request.urlopen(f"{APERTUS_BASE_URL}/models", timeout=2)
        return True
    except Exception:
        return False

if not _apertus_alive():
    print("[Apertus] serveur LoRA pas détecté — lancement en subprocess…")
    _apertus_log = open("/tmp/apertus_serve.log", "ab")
    apertus_proc = subprocess.Popen(
        ["/mnt/nfs/llmhub/sft_env/bin/python",
         "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/serve_lora.py"],
        stdout=_apertus_log, stderr=_apertus_log,
    )
    for _i in range(120):
        if _apertus_alive():
            print(f"[Apertus] ready après {_i+1}s (PID {apertus_proc.pid})")
            break
        time.sleep(1)
    else:
        print(f"[Apertus] ⚠️ pas prêt après 120s — voir /tmp/apertus_serve.log")
else:
    print("[Apertus] déjà actif sur :8001")

LOCAL_TEST_DIR = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests"

bio_hint_llm = ChatOpenAI(
    model=APERTUS_MODEL,
    base_url=APERTUS_BASE_URL,
    api_key="EMPTY",
    temperature=0.3,
    max_tokens=512,
    max_retries=1,
)

agent = BioAgent(
    path=LOCAL_TEST_DIR,
    llm=MAIN_MODEL,
    source="Custom",
    base_url=MAIN_BASE_URL,
    api_key=DEEPSEEK_KEY,
    bio_hint_llm=bio_hint_llm,
    use_tool_retriever=True,
    auto_start_artifacts=True,
    timeout_seconds=300,
)

if hasattr(agent, "llm"):
    agent.llm.max_retries = 3
    print(f"\nProvider main : DeepSeek  |  Modele : {MAIN_MODEL}")
    print(f"Provider bio_hint : LOCAL  |  Modele : {APERTUS_MODEL}")
    print(f"bio_hint activated : {agent.bio_hint_llm is not None}")

# ─── Lancement de l'UI web (port 8911) en sous-process isolé ──────────────────
UI_DIR = "/mnt/nfs/llmhub/MetaLm/sockett/agent-ui"
_ui_env = os.environ.copy()
_ui_env["AGENT_COPILOT_SECRET"] = "test-local-secret-32chars-minimum!"
_ui_env["ALLOWED_ORIGIN"]       = "*"
_ui_env["PYTHONPATH"] = "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src" + os.pathsep + _ui_env.get("PYTHONPATH", "")

try:
    subprocess.run(["pkill", "-f", "uvicorn app.main:app.*8911"], timeout=5)
    time.sleep(0.5)
except Exception:
    pass

_ui_log = open(os.path.join(UI_DIR, "ui_uvicorn.log"), "ab")
ui_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8911", "--log-level", "warning"],
    cwd=UI_DIR,
    env=_ui_env,
    stdout=_ui_log,
    stderr=_ui_log,
)

import requests as _req
for _i in range(25):
    try:
        if _req.get("http://localhost:8911/", timeout=2).status_code in (200, 401, 404):
            print(f"UI Web   : http://<IP-serveur>:8911   (PID subprocess: {ui_proc.pid})")
            break
    except Exception:
        time.sleep(1)
else:
    print(f"UI Web   : ⚠️  pas démarrée — voir {UI_DIR}/ui_uvicorn.log")

print(f"Artifacts: http://127.0.0.1:8910/docs")
print(f"Kernel CWD: {os.getcwd()}   (doit rester /mnt/nfs/llmhub/MetaLm)")


/mnt/nfs/llmhub/sft_env/lib/python3.10/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


[Apertus] serveur LoRA pas détecté — lancement en subprocess…
[Apertus] ready après 73s (PID 1364089)

BioAgent_v3 CONFIGURATION
EFFECTIVE CONFIG :
  Path: /mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests
  Run Dir: .genomeer_runs
  Timeout Seconds: 300
  Llm: deepseek-v4-pro
  Temperature: 0.7
  Use Tool Retriever: True
  Source: Custom
  Base Url: https://api.deepseek.com/v1
  Api Key: ********4ae5

LangSmith tracing: ON (project=genomeer-benchmark)

Provider main : DeepSeek  |  Modele : deepseek-v4-pro
Provider bio_hint : LOCAL  |  Modele : Apertus-8B-BioInstruct
bio_hint activated : True


INFO:     Started server process [1362329]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


Artifacts service on http://127.0.0.1:8910/docs
UI Web   : http://<IP-serveur>:8911   (PID subprocess: 1365607)
Artifacts: http://127.0.0.1:8910/docs
Kernel CWD: /mnt/nfs/llmhub/MetaLm   (doit rester /mnt/nfs/llmhub/MetaLm)


In [ ]:
# Dans une nouvelle cellule du notebook
import requests
r = requests.post(
    "https://api.cerebras.ai/v1/chat/completions",
    headers={"Authorization": ""},
    json={"model": "gpt-oss-120b", "messages": [{"role":"user","content":"hi"}], "max_tokens": 50}
)
print(r.status_code, r.text[:300])


200 {"id":"chatcmpl-4ebf1daa-9a80-4674-b3ff-1dd39192d885","choices":[{"finish_reason":"stop","index":0,"message":{"content":"Hello! How can I help you today?","reasoning":"We need to respond to a simple greeting. Should greet back, perhaps ask how can help. No special constraints.","role":"assistant"}}]


INFO:     10.52.88.105:55158 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:54392 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:43052 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:55790 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:56236 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:42866 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:36610 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:36992 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:43316 - "POST /api/sessions/4/cancel HTTP/1.1" 200 OK
INFO:     10.52.88.105:40298 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:55254 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:34092 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:57712 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:56656 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:57912 - "GET /docs HTTP/1.1" 200 OK


In [4]:
import requests, json

UI = "http://localhost:8911"

# Login avec ton compte UI
r = requests.post(f"{UI}/api/auth/login",
    data={"username": "yousseflamrabi@gmail.com", "password": "youssef"})
print("Login:", r.status_code)
TOKEN = r.json()["access_token"]

# Stream test sur session 4
r = requests.post(f"{UI}/api/sessions/4/messages",
    headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"},
    json={"message": "hi", "stream": True},
    stream=True, timeout=120)
print("POST status:", r.status_code)
for i, line in enumerate(r.iter_lines()):
    if line: print(f"[{i}]", line[:300].decode())
    if i > 30: break


INFO:     127.0.0.1:57734 - "POST /api/auth/login HTTP/1.1" 200 OK
Login: 200
INFO:     127.0.0.1:57738 - "POST /api/sessions/4/messages HTTP/1.1" 200 OK
POST status: 200
INFO:     10.52.88.105:59022 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:37242 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:59762 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:58926 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:44832 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:59578 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:37446 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:36416 - "GET /docs HTTP/1.1" 200 OK


KeyboardInterrupt: 

In [3]:
result = agent.go("hello")
print(result)

Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

hello
================================== Ai Message ==================================

<log><route>qa</route></log>
================================== Ai Message ==================================

Hello! I'm the QA node in the MetaGenomics assistant pipeline. How can I help you today? If you have any questions about data validation, assembly quality checks, or results from a previous step, feel free to ask.
(['================================\x1b Human Message \x1b=================================\n\nhello', '==================================\x1b Ai Message \x1b==================================\n\n<log><route>qa</route></log>', "==================================\x1b Ai Message \x1b==================================\n\nHello! I'm the QA node in the MetaGenomics assistant pipeline. How can I help you today? If you have any questions about data validatio

In [ ]:

# ==============================================================
# OPENROUTER — Init BioAgent
# Modele : openai/gpt-oss-120b:free
# 120B | 131k context | OpenAI open-source, format strict OK
# ==============================================================
import sys, os

os.environ['OPENROUTER_API_KEY'] = ''

from genomeer.agent.v2 import BioAgent

MODEL    = 'openai/gpt-oss-120b:free'
BASE_URL = 'https://openrouter.ai/api/v1'
API_KEY  = os.environ['OPENROUTER_API_KEY']

WSL_TEST_DIR = '/mnt/c/Users/PC/Downloads/agent genommer/agent/genomeer/tests'

agent = BioAgent(
    path=WSL_TEST_DIR,
    llm=MODEL,
    source='Custom',
    base_url=BASE_URL,
    api_key=API_KEY,
    use_tool_retriever=True,
    timeout_seconds=600,
    auto_start_artifacts=True,
    interaction_mode='auto',
)

print(f'Kernel   : {sys.executable}')
print(f'Provider : OpenRouter')
print(f'Model    : {MODEL}')
print(f'Path     : {WSL_TEST_DIR}')


/home/youssef/.local/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



BioAgent_v1 CONFIGURATION
EFFECTIVE CONFIG :
  Path: /mnt/c/Users/PC/Downloads/agent genommer/agent/genomeer/tests
  Run Dir: .genomeer_runs
  Timeout Seconds: 600
  Llm: openai/gpt-oss-120b:free
  Temperature: 0.7
  Use Tool Retriever: True
  Source: Custom
  Base Url: https://openrouter.ai/api/v1
  Api Key: ********7a7a

Kernel   : /usr/bin/python3
Provider : OpenRouter
Model    : openai/gpt-oss-120b:free
Path     : /mnt/c/Users/PC/Downloads/agent genommer/agent/genomeer/tests


INFO:     Started server process [222577]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


Artifacts service on http://127.0.0.1:8910/docs


In [ ]:
import sys, os
# sys.path.insert(0, "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/src")
# from genomeer.agent.v2 import BioAgent

# LOCAL_API_URL = "http://10.52.88.106:11434/v1"
# MODEL_ALIAS   = "gemma4:26b"

# agent = BioAgent(
#     path="/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests",
#     llm=MODEL_ALIAS,
#     source="Custom",
#     base_url=LOCAL_API_URL,
#     api_key="EMPTY",
#     use_tool_retriever=True,
#     timeout_seconds=600,
#     auto_start_artifacts=True,
#     interaction_mode="auto",
# )


In [ ]:
response = agent.go(
    "hello "
)
print(response)


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

hello 
INFO:     10.52.88.105:34652 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:52532 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:49736 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:39772 - "GET /docs HTTP/1.1" 200 OK
================================== Ai Message ==================================

<log><route>qa</route></log>
INFO:     10.52.88.105:40312 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:36146 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:52962 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:46366 - "GET /docs HTTP/1.1" 200 OK
================================== Ai Message ==================================

Hello! How can I assist you today?
(['================================\x1b Human Message \x1b=================================\n\nhello ', '==================================\x1b A

INFO:     10.52.88.105:58308 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:41182 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:59176 - "POST /api/sessions/4/messages HTTP/1.1" 200 OK
INFO:     10.52.88.105:59190 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:53966 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:43524 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:34602 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:36096 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:40050 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:41024 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:47704 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:55302 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:48304 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:49086 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:34314 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:51174 - "GET /api/auth/me HTTP/1.1" 200 OK
I

In [ ]:
response = agent.go(
    "In a single pipeline: "
    "1) Download Bacillus subtilis 168 genome from NCBI using accession GCF_000009045.1 "
    "and E. coli K12 genome using accession GCF_000005845.2 — both in the same output folder. "
    "2) For each downloaded genome compute: total length, GC content, number of contigs, N50. "
    "3) Create a side-by-side bar chart with matplotlib comparing the two genomes "
    "and save it as genome_comparison.png. "
    "4) Save a summary table to comparison_report.txt."
)
print(response)



Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

In a single pipeline: 1) Download Bacillus subtilis 168 genome from NCBI using accession GCF_000009045.1 and E. coli K12 genome using accession GCF_000005845.2 — both in the same output folder. 2) For each downloaded genome compute: total length, GC content, number of contigs, N50. 3) Create a side-by-side bar chart with matplotlib comparing the two genomes and save it as genome_comparison.png. 4) Save a summary table to comparison_report.txt.
================================== Ai Message ==================================

- [ ] Use ncbi-genome-download with assembly accessions GCF_000009045.1,GCF_000005845.2 (RefSeq bacteria) to download both genomes into the run directory and decompress the resulting .fna.gz files.
- [ ] Run seqkit stats on the Bacillus subtilis FASTA file and the Escherichia coli FASTA file, saving results as seqkit_stats_bsubtilis.tsv

INFO:     127.0.0.1:53860 - "POST /api/v1/artifacts/runs/create/20 HTTP/1.1" 200 OK
INFO:     127.0.0.1:53868 - "POST /api/v1/artifacts/runs/20/upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:53884 - "POST /api/v1/artifacts/runs/20/publish HTTP/1.1" 200 OK
INFO:     127.0.0.1:50106 - "POST /api/v1/artifacts/runs/create/20 HTTP/1.1" 200 OK
INFO:     127.0.0.1:50120 - "POST /api/v1/artifacts/runs/20/upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:50122 - "POST /api/v1/artifacts/runs/20/publish HTTP/1.1" 200 OK
INFO:     127.0.0.1:51372 - "POST /api/v1/artifacts/runs/create/20 HTTP/1.1" 200 OK
INFO:     127.0.0.1:51388 - "POST /api/v1/artifacts/runs/20/upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:51394 - "POST /api/v1/artifacts/runs/20/publish HTTP/1.1" 200 OK
INFO:     127.0.0.1:52104 - "POST /api/v1/artifacts/runs/create/21 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52112 - "POST /api/v1/artifacts/runs/21/upload HTTP/1.1" 200 OK
INFO:     127.0.0.1:52114 - "POST /api/v1/artifacts/runs/21/publish HTTP/

In [3]:
response = agent.go(
    "Given a FASTA file of metagenomic contigs, run a basic metagenomics analysis pipeline: "
    "1) Compute assembly statistics (number of contigs, total length, GC content, N50) using Python stdlib only. "
    "2) Predict ORFs on the contigs using Prodigal and save the protein sequences to predicted_proteins.faa. "
    "3) Count the number of predicted proteins and calculate the average protein length. "
    "4) Save a summary report to metagenomics_report.txt. "
    "Use this synthetic FASTA as input — generate it yourself with 5 random contigs of 1000-5000 bp each."
)

Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Given a FASTA file of metagenomic contigs, run a basic metagenomics analysis pipeline: 1) Compute assembly statistics (number of contigs, total length, GC content, N50) using Python stdlib only. 2) Predict ORFs on the contigs using Prodigal and save the protein sequences to predicted_proteins.faa. 3) Count the number of predicted proteins and calculate the average protein length. 4) Save a summary report to metagenomics_report.txt. Use this synthetic FASTA as input — generate it yourself with 5 random contigs of 1000-5000 bp each.
================================== Ai Message ==================================

- [ ] Generate a synthetic FASTA file with 5 random DNA contigs (length 1000‑5000 bp) and save as synthetic_contigs.fna  
- [ ] Compute assembly statistics (contig count, total length, GC%, N50) from synthetic_contigs.fna using only Python standard 

## test 3

In [5]:
response = agent.go(
    "Run a lightweight metagenomics analysis on the FASTA file at "
    "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/02-local-fasta/sequence_tiny.fasta :\n"
    "1) Use seqkit stats -a to compute assembly statistics (N50, GC%, contig count, total length) "
    "and save the output to seqkit_stats.tsv.\n"
    "2) Run Prodigal in metagenome mode (-p meta) to predict ORFs and save protein sequences "
    "to predicted_proteins.faa and gene coordinates to genes.gff.\n"
    "3) Parse the predicted_proteins.faa with Python stdlib (no Bio) to count total proteins "
    "and compute average protein length.\n"
    "4) Save a final summary report to pipeline_report.txt with: "
    "seqkit stats results, protein count, and average protein length."
)

Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Run a lightweight metagenomics analysis on the FASTA file at /mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/02-local-fasta/sequence_tiny.fasta :
1) Use seqkit stats -a to compute assembly statistics (N50, GC%, contig count, total length) and save the output to seqkit_stats.tsv.
2) Run Prodigal in metagenome mode (-p meta) to predict ORFs and save protein sequences to predicted_proteins.faa and gene coordinates to genes.gff.
3) Parse the predicted_proteins.faa with Python stdlib (no Bio) to count total proteins and compute average protein length.
4) Save a final summary report to pipeline_report.txt with: seqkit stats results, protein count, and average protein length.
================================== Ai Message ==================================

- [ ] Compute assembly statistics with `seqkit stats -a` on the provided FASTA and write results to `seqkit_st

INFO:     127.0.0.1:58198 - "GET /api/v1/artifacts/download/ef40cbbe-64b3-4be4-a007-700afefaa3ab/outputs/pipeline_report.txt HTTP/1.1" 200 OK


## test 4-1

In [2]:
response = agent.go(
    "Run a complete metagenomics analysis on the real FASTA file at "
    "/mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/02-local-fasta/sequence_tiny.fasta:\n"
    "1) Use seqkit stats -a to compute assembly statistics (total length, GC%, N50, contig count) "
    "and save to seqkit_stats.tsv.\n"
    "2) Run Prodigal in metagenome mode (-p meta) to predict ORFs, save proteins to "
    "predicted_proteins.faa and gene coordinates to genes.gff.\n"
    "3) Parse predicted_proteins.faa with Python stdlib to count total proteins "
    "and compute average protein length in bp.\n"
    "4) Parse genes.gff to compute ORF density (ORFs per kb).\n"
    "5) Save a final report to analysis_report.txt with: "
    "seqkit stats, protein count, average protein length, ORF density."
)
print(response)


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Run a complete metagenomics analysis on the real FASTA file at /mnt/nfs/llmhub/MetaLm/sockett/genomeer/tests/02-local-fasta/sequence_tiny.fasta:
1) Use seqkit stats -a to compute assembly statistics (total length, GC%, N50, contig count) and save to seqkit_stats.tsv.
2) Run Prodigal in metagenome mode (-p meta) to predict ORFs, save proteins to predicted_proteins.faa and gene coordinates to genes.gff.
3) Parse predicted_proteins.faa with Python stdlib to count total proteins and compute average protein length in bp.
4) Parse genes.gff to compute ORF density (ORFs per kb).
5) Save a final report to analysis_report.txt with: seqkit stats, protein count, average protein length, ORF density.
================================== Ai Message ==================================

- [ ] Compute assembly statistics with `seqkit stats -a` on the FASTA file and write resu

## test 5

In [3]:
response = agent.go(
    "Full metagenomics QC pipeline — download and analyse a real genome:\n"
    "1) Download the Mycoplasma genitalium G37 genome"
    "2) Run seqkit stats -a --tabular on the downloaded FASTA\n"
    "3) Run quast.py on the FASTA (no reference) and save output to quast_output/ folder.\n"
    "4) Run Prodigal -p meta on the FASTA to predict ORFs, save proteins to predicted_proteins.faa "
    "and coordinates to genes.gff.\n"
    "5) count predicted_proteins.faa  total proteins "
    "and compute average protein length (in amino acids).\n"
    "6) Write a final report summary.txt containing: "
    "genome accession, total length, GC%, N50 (from seqkit), "
    "QUAST N50 and contig count (from quast_output/report.tsv), "
    "protein count and average protein length."
)
print(response)


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Full metagenomics QC pipeline — download and analyse a real genome:
1) Download the Mycoplasma genitalium G37 genome2) Run seqkit stats -a --tabular on the downloaded FASTA
3) Run quast.py on the FASTA (no reference) and save output to quast_output/ folder.
4) Run Prodigal -p meta on the FASTA to predict ORFs, save proteins to predicted_proteins.faa and coordinates to genes.gff.
5) count predicted_proteins.faa  total proteins and compute average protein length (in amino acids).
6) Write a final report summary.txt containing: genome accession, total length, GC%, N50 (from seqkit), QUAST N50 and contig count (from quast_output/report.tsv), protein count and average protein length.
================================== Ai Message ==================================

- [ ] Download the Mycoplasma genitalium G37 genome (accession GCF_000008565.1) using `ncbi-genome

## test 6

In [2]:
response = agent.go(
    "Download the Salmonella enterica Typhimurium LT2 reference genome from NCBI RefSeq "
    "and run a complete quality assessment: assembly statistics, gene prediction, and "
    "write a final report with all key metrics."
)
print(response)


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Download the Salmonella enterica Typhimurium LT2 reference genome from NCBI RefSeq and run a complete quality assessment: assembly statistics, gene prediction, and write a final report with all key metrics.
================================== Ai Message ==================================

- [ ] Download the *Salmonella enterica* Typhimurium LT2 reference genome (assembly accession GCF_000006945.2) using `ncbi-genome-download` with the correct flags and decompress the resulting `.fna.gz` file.  
- [ ] Compute assembly statistics on the downloaded FASTA with `seqkit stats -a --tabular`, saving the tab‑separated output to `seqkit_stats.tsv`.  
- [ ] Run `quast.py` on the FASTA (no reference) to generate a full assembly quality report in the directory `quast_output/`.  
- [ ] Predict genes on the FASTA with Prodigal (default bacterial mode) to produce `predicte

## test 7

In [3]:
response = agent.go(
    "Download E. coli K-12 genome from NCBI RefSeq, annotate it completely, "
    "and screen it for antibiotic resistance genes. Write a summary report."
)
print(response)


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Download E. coli K-12 genome from NCBI RefSeq, annotate it completely, and screen it for antibiotic resistance genes. Write a summary report.
================================== Ai Message ==================================

- [ ] Use ncbi-genome-download with the accession GCF_000005845.2 to fetch the RefSeq Escherichia coli K‑12 genome into the run directory  
- [ ] Decompress the downloaded *.fna.gz file to obtain a plain FASTA genome file (e.g., GCF_000005845.2.fna)  
- [ ] Run seqkit stats on the FASTA genome and write the tabular results to seqkit_stats.tsv  
- [ ] Run Prodigal on the FASTA genome to predict genes, producing predicted_proteins.faa and genes.gff  
- [ ] Run abricate on the genome FASTA (or genes.gff) using the CARD database and write the results to abricate_card.tsv  
- [ ] Parse seqkit_stats.tsv, Prodigal gene count, and abricate_card

## full pipeline test

In [2]:
result = agent.go("""
Full metagenomics pipeline on simulated reads:

1) Download Salmonella enterica LT2 genome (GCF_000006945.2) from NCBI RefSeq as reference.
2) Simulate 50,000 paired-end Illumina reads (150bp) from the genome using wgsim: wgsim -N 50000 -1 150 -2 150 genome.fna reads_R1.fastq reads_R2.fastq.
3) Run FastQC on reads_R1.fastq and reads_R2.fastq, save reports to fastqc_raw/.
4) Trim adapters and low-quality bases with fastp (min quality 20, min length 50), output trimmed_R1.fastq and trimmed_R2.fastq, save fastp.json and fastp.html.
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 200), output to megahit_output/.
6) Run seqkit stats -a --tabular on megahit_output/final.contigs.fa, save to assembly_stats.tsv.
7) Run quast.py on final.contigs.fa (no reference), save to quast_output/.
8) Map trimmed reads back to contigs using minimap2 (-ax sr), sort and index BAM with samtools, compute coverage depth with jgi_summarize_bam_contig_depths, save depth.txt.
9) Bin contigs with MetaBAT2 using depth.txt (--minContig 200), output prefix bins/bin.
10) Run Prodigal -p meta on final.contigs.fa, save predicted_proteins.faa and genes.gff.
11) Run abricate --db card on final.contigs.fa, save abricate_card.tsv.
12) Write summary.txt with: total contigs, N50, GC%, number of bins, protein count, AMR gene count.
""")

print(result)

INFO:     Started server process [2243050]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


Artifacts service on http://127.0.0.1:8910/docs
Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================


Full metagenomics pipeline on simulated reads:

1) Download Salmonella enterica LT2 genome (GCF_000006945.2) from NCBI RefSeq as reference.
2) Simulate 50,000 paired-end Illumina reads (150bp) from the genome using wgsim: wgsim -N 50000 -1 150 -2 150 genome.fna reads_R1.fastq reads_R2.fastq.
3) Run FastQC on reads_R1.fastq and reads_R2.fastq, save reports to fastqc_raw/.
4) Trim adapters and low-quality bases with fastp (min quality 20, min length 50), output trimmed_R1.fastq and trimmed_R2.fastq, save fastp.json and fastp.html.
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 200), output to megahit_output/.
6) Run seqkit stats -a --tabular on megahit_output/final.contigs.fa, save to assembly_stats.tsv.
7) Run quast.py on final.contigs.fa (no reference), save to quast_output/.
8) Map trimmed reads b

In [2]:
response=agent.go("""Taxonomic classification of simulated viral reads:

1) Download SARS-CoV-2 reference genome (GCF_009858895.2) from NCBI RefSeq.
2) Simulate 30,000 paired-end Illumina reads (150bp) from the genome using wgsim, save as reads_R1.fastq and reads_R2.fastq.
3) Trim adapters with fastp (--disable_quality_filtering, --length_required 50), output trimmed_R1.fastq and trimmed_R2.fastq.
4) Run Kraken2 on the trimmed paired-end reads using the viral database installed at /mnt/nfs/llmhub/kraken2_db, save kraken2.report and kraken2.out.
5) Run Bracken on kraken2.report at the species level (-l S, -r 150), save bracken_species.tsv.
6) Write summary.txt with: total reads classified, unclassified %, top 3 species by abundance, total Bracken estimated reads.""")
print(response)

INFO:     Started server process [3182442]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8910 (Press CTRL+C to quit)


Artifacts service on http://127.0.0.1:8910/docs
Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Taxonomic classification of simulated viral reads:

1) Download SARS-CoV-2 reference genome (GCF_009858895.2) from NCBI RefSeq.
2) Simulate 30,000 paired-end Illumina reads (150bp) from the genome using wgsim, save as reads_R1.fastq and reads_R2.fastq.
3) Trim adapters with fastp (--disable_quality_filtering, --length_required 50), output trimmed_R1.fastq and trimmed_R2.fastq.
4) Run Kraken2 on the trimmed paired-end reads using the viral database installed at /mnt/nfs/llmhub/kraken2_db, save kraken2.report and kraken2.out.
5) Run Bracken on kraken2.report at the species level (-l S, -r 150), save bracken_species.tsv.
6) Write summary.txt with: total reads classified, unclassified %, top 3 species by abundance, total Bracken estimated reads.
================================== Ai Message ======================

In [2]:
agent.go("""Metagenomics binning pipeline:

1) Download Salmonella enterica LT2 (GCF_000006945.2) and Escherichia coli K-12 (GCF_000005845.2) from NCBI RefSeq.
2) Concatenate both genomes into mixed.fna.
3) Simulate 500,000 paired-end Illumina reads (150bp) from mixed.fna using wgsim (gives ~15x coverage for proper binning).
4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.
6) Map reads back with minimap2, sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths.
7) Bin contigs with MetaBAT2 (-m 1500), output bins/bin.
8) Write summary.txt with: number of bins, mean bin size (bp), N50 of contigs, total bases binned, % of assembly placed into bins.""")


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Metagenomics binning pipeline:

1) Download Salmonella enterica LT2 (GCF_000006945.2) and Escherichia coli K-12 (GCF_000005845.2) from NCBI RefSeq.
2) Concatenate both genomes into mixed.fna.
3) Simulate 500,000 paired-end Illumina reads (150bp) from mixed.fna using wgsim (gives ~15x coverage for proper binning).
4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.
6) Map reads back with minimap2, sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths.
7) Bin contigs with MetaBAT2 (-m 1500), output bins/bin.
8) Write summary.txt with: number of bins, mean bin size (bp), N50 of contigs, total bases binned, % of assembly placed into bins.
================================== Ai Message ==================================

- [ ] D

(['================================\x1b Human Message \x1b=================================\n\nMetagenomics binning pipeline:\n\n1) Download Salmonella enterica LT2 (GCF_000006945.2) and Escherichia coli K-12 (GCF_000005845.2) from NCBI RefSeq.\n2) Concatenate both genomes into mixed.fna.\n3) Simulate 500,000 paired-end Illumina reads (150bp) from mixed.fna using wgsim (gives ~15x coverage for proper binning).\n4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).\n5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.\n6) Map reads back with minimap2, sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths.\n7) Bin contigs with MetaBAT2 (-m 1500), output bins/bin.\n8) Write summary.txt with: number of bins, mean bin size (bp), N50 of contigs, total bases binned, % of assembly placed into bins.',
  '==================================\x1b Ai Message \x1b==================================\n\n- [ ] Download Salmo

In [2]:
agent.go("""Variant calling pipeline:

1) Download Escherichia coli K-12 MG1655 (GCF_000005845.2) from NCBI RefSeq as reference.
2) Simulate 40,000 paired-end Illumina reads (150bp) from the reference with mutation rate 0.001 using wgsim (-r 0.001).
3) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
4) Build a bwa index on the reference genome.
5) Map trimmed reads with bwa mem, pipe to samtools to produce sorted indexed BAM.
6) Call variants with bcftools mpileup + bcftools call -mv, save variants.vcf.
7) Filter variants with bcftools filter (QUAL>20, DP>10), save variants_filtered.vcf.
8) Write summary.txt with: total reads mapped, % mapping rate, total raw variants, total filtered variants, SNPs vs indels count.""")
#pour ls tools manquante on l'installe avec pip si possibble tool par tool et aussi pour les databases on les laisse pour le moment on install juste cell de hmm je pense il faut l'installer ppour passer le test cell de hmm n'est ce pas 


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Variant calling pipeline:

1) Download Escherichia coli K-12 MG1655 (GCF_000005845.2) from NCBI RefSeq as reference.
2) Simulate 40,000 paired-end Illumina reads (150bp) from the reference with mutation rate 0.001 using wgsim (-r 0.001).
3) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
4) Build a bwa index on the reference genome.
5) Map trimmed reads with bwa mem, pipe to samtools to produce sorted indexed BAM.
6) Call variants with bcftools mpileup + bcftools call -mv, save variants.vcf.
7) Filter variants with bcftools filter (QUAL>20, DP>10), save variants_filtered.vcf.
8) Write summary.txt with: total reads mapped, % mapping rate, total raw variants, total filtered variants, SNPs vs indels count.
================================== Ai Message ==================================

- [ ] Download the E. coli K‑12 MG1655 referen

(['================================\x1b Human Message \x1b=================================\n\nVariant calling pipeline:\n\n1) Download Escherichia coli K-12 MG1655 (GCF_000005845.2) from NCBI RefSeq as reference.\n2) Simulate 40,000 paired-end Illumina reads (150bp) from the reference with mutation rate 0.001 using wgsim (-r 0.001).\n3) Trim reads with fastp (--disable_quality_filtering, --length_required 50).\n4) Build a bwa index on the reference genome.\n5) Map trimmed reads with bwa mem, pipe to samtools to produce sorted indexed BAM.\n6) Call variants with bcftools mpileup + bcftools call -mv, save variants.vcf.\n7) Filter variants with bcftools filter (QUAL>20, DP>10), save variants_filtered.vcf.\n8) Write summary.txt with: total reads mapped, % mapping rate, total raw variants, total filtered variants, SNPs vs indels count.',
  '==================================\x1b Ai Message \x1b==================================\n\n- [ ] Download the E. coli K‑12 MG1655 reference (GCF_00000

In [2]:
agent.go("""Protein homology search pipeline:

1) Download Escherichia coli K-12 (GCF_000005845.2) genome and proteins from NCBI RefSeq.
2) Predict ORFs on the genome with Prodigal (-f gff, single mode), save predicted_proteins.faa and genes.gff.
3) Build a BLAST protein database from the downloaded reference proteins using makeblastdb (-dbtype prot), save blast_db/.
4) Run blastp on predicted_proteins.faa against blast_db with -outfmt 6 -evalue 1e-10, save blastp_results.tsv.
5) Build a DIAMOND database from the reference proteins (diamond makedb), save diamond_db.dmnd.
6) Run diamond blastp on predicted_proteins.faa against diamond_db (--outfmt 6 -e 1e-10), save diamond_results.tsv.
7) Write summary.txt with: total predicted proteins, BLAST hits count, DIAMOND hits count, mean % identity, runtime comparison.""")


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Protein homology search pipeline:

1) Download Escherichia coli K-12 (GCF_000005845.2) genome and proteins from NCBI RefSeq.
2) Predict ORFs on the genome with Prodigal (-f gff, single mode), save predicted_proteins.faa and genes.gff.
3) Build a BLAST protein database from the downloaded reference proteins using makeblastdb (-dbtype prot), save blast_db/.
4) Run blastp on predicted_proteins.faa against blast_db with -outfmt 6 -evalue 1e-10, save blastp_results.tsv.
5) Build a DIAMOND database from the reference proteins (diamond makedb), save diamond_db.dmnd.
6) Run diamond blastp on predicted_proteins.faa against diamond_db (--outfmt 6 -e 1e-10), save diamond_results.tsv.
7) Write summary.txt with: total predicted proteins, BLAST hits count, DIAMOND hits count, mean % identity, runtime comparison.
================================== Ai Message ============

(['================================\x1b Human Message \x1b=================================\n\nProtein homology search pipeline:\n\n1) Download Escherichia coli K-12 (GCF_000005845.2) genome and proteins from NCBI RefSeq.\n2) Predict ORFs on the genome with Prodigal (-f gff, single mode), save predicted_proteins.faa and genes.gff.\n3) Build a BLAST protein database from the downloaded reference proteins using makeblastdb (-dbtype prot), save blast_db/.\n4) Run blastp on predicted_proteins.faa against blast_db with -outfmt 6 -evalue 1e-10, save blastp_results.tsv.\n5) Build a DIAMOND database from the reference proteins (diamond makedb), save diamond_db.dmnd.\n6) Run diamond blastp on predicted_proteins.faa against diamond_db (--outfmt 6 -e 1e-10), save diamond_results.tsv.\n7) Write summary.txt with: total predicted proteins, BLAST hits count, DIAMOND hits count, mean % identity, runtime comparison.',
  '==================================\x1b Ai Message \x1b============================

In [ ]:
! grep -c "pingHealth\|hb-backend" /mnt/nfs/llmhub/MetaLm/sockett/agent-ui/static/js/app.js


0


INFO:     10.52.88.105:54726 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:56834 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:51286 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:59866 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:40256 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:33342 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:38910 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:52258 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:53948 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:51764 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:46370 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:57662 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:40036 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:45572 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:56202 - "GET /api/auth/me HTTP/1.1" 200 OK
INFO:     127.0.0.1:43100 - "GET /docs HTTP/1.1" 200 OK
INFO:     10.52.88.105:5

In [ ]:
agent.go("""HMM profile search pipeline:

1) Download Salmonella enterica LT2 (GCF_000006945.2) from NCBI RefSeq.
2) Predict ORFs with Prodigal (single mode, -f gff), save predicted_proteins.faa.
3) Download a small HMM profile set (e.g. Pfam ribosomal proteins or use a locally available reference HMM).
4) Run hmmpress on the HMM file to prepare the database.
5) Run hmmscan with --tblout hmmscan_results.tsv on predicted_proteins.faa against the HMM database.
6) Parse hmmscan_results.tsv to count hits per profile and unique proteins with at least one significant hit (E-value < 1e-5).
7) Write summary.txt with: total predicted proteins, proteins with ≥1 HMM hit, top 5 HMM profiles by hit count.""")


In [2]:
agent.go("""Phylogeny pipeline:

1) Download 4 bacterial genomes from NCBI RefSeq: E. coli K-12 (GCF_000005845.2), Salmonella LT2 (GCF_000006945.2), Shigella flexneri (GCF_000006925.2), Klebsiella pneumoniae (GCF_000240185.1).
2) Predict 16S rRNA on each genome with barrnap (--kingdom bac), extract 16S sequences into 16S.fasta.
3) Align the 16S sequences with mafft --auto, save 16S_aligned.fasta.
4) Trim the alignment with trimal -automated1, save 16S_trimmed.fasta.
5) Build a phylogenetic tree with FastTree -nt -gtr on 16S_trimmed.fasta, save tree.nwk.
6) Write summary.txt with: number of 16S extracted per genome, alignment length before/after trimal, tree in Newick format.""")


================================ Human Message =================================

Phylogeny pipeline:

1) Download 4 bacterial genomes from NCBI RefSeq: E. coli K-12 (GCF_000005845.2), Salmonella LT2 (GCF_000006945.2), Shigella flexneri (GCF_000006925.2), Klebsiella pneumoniae (GCF_000240185.1).
2) Predict 16S rRNA on each genome with barrnap (--kingdom bac), extract 16S sequences into 16S.fasta.
3) Align the 16S sequences with mafft --auto, save 16S_aligned.fasta.
4) Trim the alignment with trimal -automated1, save 16S_trimmed.fasta.
5) Build a phylogenetic tree with FastTree -nt -gtr on 16S_trimmed.fasta, save tree.nwk.
6) Write summary.txt with: number of 16S extracted per genome, alignment length before/after trimal, tree in Newick format.


KeyboardInterrupt: 

In [2]:
agent.go("""Bacterial genome assembly and annotation:

1) Download Staphylococcus aureus NCTC 8325 (GCF_000013425.1) from NCBI RefSeq.
2) Simulate 60,000 paired-end Illumina reads (150bp) from the genome using wgsim.
3) Trim with fastp (--disable_quality_filtering, --length_required 50).
4) Assemble trimmed reads with SPAdes (--only-assembler, -k 21,33,55,77), save assembly to spades_output/.
5) Run seqkit stats -a --tabular on spades_output/contigs.fasta, save assembly_stats.tsv.
6) Annotate assembly with Prokka (--kingdom Bacteria, --genus Staphylococcus), save prokka_output/.
7) Write summary.txt with: total contigs, N50, GC%, total CDS, total tRNA, total rRNA from Prokka.""")


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Bacterial genome assembly and annotation:

1) Download Staphylococcus aureus NCTC 8325 (GCF_000013425.1) from NCBI RefSeq.
2) Simulate 60,000 paired-end Illumina reads (150bp) from the genome using wgsim.
3) Trim with fastp (--disable_quality_filtering, --length_required 50).
4) Assemble trimmed reads with SPAdes (--only-assembler, -k 21,33,55,77), save assembly to spades_output/.
5) Run seqkit stats -a --tabular on spades_output/contigs.fasta, save assembly_stats.tsv.
6) Annotate assembly with Prokka (--kingdom Bacteria, --genus Staphylococcus), save prokka_output/.
7) Write summary.txt with: total contigs, N50, GC%, total CDS, total tRNA, total rRNA from Prokka.
================================== Ai Message ==================================

- [ ] Download Staphylococcus aureus NCTC 8325 genome (assembly accession GCF_000013425.1) using ncbi-genome-down

(['================================\x1b Human Message \x1b=================================\n\nBacterial genome assembly and annotation:\n\n1) Download Staphylococcus aureus NCTC 8325 (GCF_000013425.1) from NCBI RefSeq.\n2) Simulate 60,000 paired-end Illumina reads (150bp) from the genome using wgsim.\n3) Trim with fastp (--disable_quality_filtering, --length_required 50).\n4) Assemble trimmed reads with SPAdes (--only-assembler, -k 21,33,55,77), save assembly to spades_output/.\n5) Run seqkit stats -a --tabular on spades_output/contigs.fasta, save assembly_stats.tsv.\n6) Annotate assembly with Prokka (--kingdom Bacteria, --genus Staphylococcus), save prokka_output/.\n7) Write summary.txt with: total contigs, N50, GC%, total CDS, total tRNA, total rRNA from Prokka.',
  '==================================\x1b Ai Message \x1b==================================\n\n- [ ] Download Staphylococcus aureus NCTC 8325 genome (assembly accession GCF_000013425.1) using ncbi-genome-download with RefS

In [3]:
agent.go("""Bin quality assessment pipeline:

1) Download Salmonella enterica LT2 (GCF_000006945.2) and Escherichia coli K-12 (GCF_000005845.2) from NCBI RefSeq.
2) Concatenate both genomes into mixed.fna.
3) Simulate 500,000 paired-end Illumina reads (150bp) from mixed.fna using wgsim (~15x coverage).
4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.
6) Map reads back with minimap2, sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths.
7) Bin contigs with MetaBAT2 (-m 1500) to bins/.
8) Run CheckM2 predict on the bins/ directory, save quality_report.tsv to checkm2_out/.
9) Write summary.txt with: number of bins, mean completeness (%), mean contamination (%), per-bin completeness and contamination breakdown.""")


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Bin quality assessment pipeline:

1) Download Salmonella enterica LT2 (GCF_000006945.2) and Escherichia coli K-12 (GCF_000005845.2) from NCBI RefSeq.
2) Concatenate both genomes into mixed.fna.
3) Simulate 500,000 paired-end Illumina reads (150bp) from mixed.fna using wgsim (~15x coverage).
4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.
6) Map reads back with minimap2, sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths.
7) Bin contigs with MetaBAT2 (-m 1500) to bins/.
8) Run CheckM2 predict on the bins/ directory, save quality_report.tsv to checkm2_out/.
9) Write summary.txt with: number of bins, mean completeness (%), mean contamination (%), per-bin completeness and contamination breakdown.
=======================

(['================================\x1b Human Message \x1b=================================\n\nBin quality assessment pipeline:\n\n1) Download Salmonella enterica LT2 (GCF_000006945.2) and Escherichia coli K-12 (GCF_000005845.2) from NCBI RefSeq.\n2) Concatenate both genomes into mixed.fna.\n3) Simulate 500,000 paired-end Illumina reads (150bp) from mixed.fna using wgsim (~15x coverage).\n4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).\n5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.\n6) Map reads back with minimap2, sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths.\n7) Bin contigs with MetaBAT2 (-m 1500) to bins/.\n8) Run CheckM2 predict on the bins/ directory, save quality_report.tsv to checkm2_out/.\n9) Write summary.txt with: number of bins, mean completeness (%), mean contamination (%), per-bin completeness and contamination breakdown.',
  '==================================\x1b Ai Mes

## benchamrk 1 categorie 1: ZymoBIOMICS D6300

In [4]:
agent.go("""Metagenomic MAG recovery benchmark:

1) Download 4 reference bacterial genomes from NCBI RefSeq into refs/: 
   - Bacillus subtilis 168 (GCF_000009045.1)
   - Escherichia coli K-12 (GCF_000005845.2)
   - Pseudomonas aeruginosa PAO1 (GCF_000006765.1)
   - Staphylococcus aureus NCTC 8325 (GCF_000013425.1)
2) Concatenate the 4 genomes into mixed.fna (this is the mock community).
3) Simulate 800,000 paired-end Illumina reads (150bp) from mixed.fna with wgsim (~15x coverage).
4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.
6) Map reads back with minimap2 (-ax sr), sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths to depth.txt.
7) Bin contigs with MetaBAT2 (-m 1500) to bins/.
8) Run CheckM2 predict on bins/ -x fa, save quality_report.tsv to checkm2_out/.
9) Identify each bin: blastn each bin against the 4 reference genomes, assign the bin to the reference with highest total alignment length and report mean %identity.
10) Write summary.txt with: number of bins, mean completeness%, mean contamination%, per-bin assignment (which reference) + mean %identity + %genome covered, list of references NOT recovered.""")


Using prompt-based retrieval with the agent's LLM
================================ Human Message =================================

Metagenomic MAG recovery benchmark:

1) Download 4 reference bacterial genomes from NCBI RefSeq into refs/: 
   - Bacillus subtilis 168 (GCF_000009045.1)
   - Escherichia coli K-12 (GCF_000005845.2)
   - Pseudomonas aeruginosa PAO1 (GCF_000006765.1)
   - Staphylococcus aureus NCTC 8325 (GCF_000013425.1)
2) Concatenate the 4 genomes into mixed.fna (this is the mock community).
3) Simulate 800,000 paired-end Illumina reads (150bp) from mixed.fna with wgsim (~15x coverage).
4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).
5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.
6) Map reads back with minimap2 (-ax sr), sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths to depth.txt.
7) Bin contigs with MetaBAT2 (-m 1500) to bins/.
8) Run CheckM2 predict on bins/ -x fa, save q

(['================================\x1b Human Message \x1b=================================\n\nMetagenomic MAG recovery benchmark:\n\n1) Download 4 reference bacterial genomes from NCBI RefSeq into refs/: \n   - Bacillus subtilis 168 (GCF_000009045.1)\n   - Escherichia coli K-12 (GCF_000005845.2)\n   - Pseudomonas aeruginosa PAO1 (GCF_000006765.1)\n   - Staphylococcus aureus NCTC 8325 (GCF_000013425.1)\n2) Concatenate the 4 genomes into mixed.fna (this is the mock community).\n3) Simulate 800,000 paired-end Illumina reads (150bp) from mixed.fna with wgsim (~15x coverage).\n4) Trim reads with fastp (--disable_quality_filtering, --length_required 50).\n5) Assemble trimmed reads with MEGAHIT (--min-contig-len 1500) to megahit_output/.\n6) Map reads back with minimap2 (-ax sr), sort/index BAM with samtools, compute depth with jgi_summarize_bam_contig_depths to depth.txt.\n7) Bin contigs with MetaBAT2 (-m 1500) to bins/.\n8) Run CheckM2 predict on bins/ -x fa, save quality_report.tsv to che

## test ui 

In [ ]:
# ============================================================
# PREFLIGHT — Verifier Ollama est lance et llama3.1:8b present
# ============================================================
import requests as _req

OLLAMA_HOST = "http://localhost:11434"

# 1. Ping Ollama
try:
    _r = _req.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    _r.raise_for_status()
    _tags = _r.json()
    print(f"OK  Ollama repond sur {OLLAMA_HOST}")
except Exception as _e:
    print(f"ECHEC  Ollama ne repond pas : {_e}")
    print("  --> Demarrer Ollama avec : ollama serve")
    raise SystemExit("Ollama non disponible — arreter ici")

# 2. Verifier que llama3.1:8b est telecharge
_models = [m["name"] for m in _tags.get("models", [])]
_target = "llama3:8b"
_found  = any(_target in m for m in _models)
print(f"Modeles disponibles : {_models}")

if _found:
    print(f"OK  '{_target}' present")
else:
    print(f"ECHEC  '{_target}' introuvable")
    print(f"  --> Telecharger avec : ollama pull {_target}")
    raise SystemExit(f"Modele {_target} manquant — arreter ici")

print("\nPREFLIGHT PASSED — pret a continuer")


OK  Ollama repond sur http://localhost:11434
Modeles disponibles : ['llama3:latest', 'phi3:latest', 'llama3:8b']
OK  'llama3:8b' present

PREFLIGHT PASSED — pret a continuer


In [1]:
_ = agent.visualize_graph(mode="manual")


NameError: name 'agent' is not defined

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 — Environnement & dépendances                    ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, os, subprocess, time

# Chemin racine du projet
UI_DIR    = r"C:\Users\PC\Downloads\agent genommer\agent\agent-ui"
AGENT_DIR = r"C:\Users\PC\Downloads\agent genommer\agent\genomeer\src"
sys.path.insert(0, AGENT_DIR)
os.chdir(UI_DIR)

# Variables d'environnement obligatoires
os.environ["AGENT_COPILOT_SECRET"]    = "test-local-secret-32chars-minimum!"
os.environ["ALLOWED_ORIGIN"]          = "http://localhost:8000"
os.environ["GENOMEER_SKIP_ENV_INSTALL"] = "1"
os.environ["TRANSFORMERS_OFFLINE"]    = "1"
os.environ["GENOMEER_RAG_OFFLINE"]    = "1"

# Installer les dépendances UI si besoin
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True, cwd=UI_DIR
)
if result.returncode != 0:
    print("PIP ERRORS:", result.stderr[-500:])
else:
    print("✅ Dépendances OK")


✅ Dépendances OK


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Démarrage du serveur FastAPI                   ║
# ╚══════════════════════════════════════════════════════════════╝
import threading, uvicorn, importlib

BASE_URL = "http://localhost:8000"

def _start_server():
    # Re-set env dans le thread uvicorn
    os.environ["AGENT_COPILOT_SECRET"] = "test-local-secret-32chars-minimum!"
    os.environ["ALLOWED_ORIGIN"]       = "http://localhost:8000"
    os.chdir(UI_DIR)
    sys.path.insert(0, UI_DIR)
    # Reload app modules proprement
    for mod in list(sys.modules.keys()):
        if mod.startswith("app"):
            del sys.modules[mod]
    uvicorn.run("app.main:app", host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

# Attendre que le serveur soit prêt
import requests as req
for i in range(20):
    try:
        r = req.get(f"{BASE_URL}/", timeout=2)
        print(f"✅ Serveur UP — status {r.status_code}  ({BASE_URL})")
        break
    except Exception:
        time.sleep(1)
        print(f"  Attente... ({i+1}/20)")
else:
    print("❌ Serveur non démarré après 20s — vérifier les erreurs ci-dessus")


c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


  Attente... (1/20)
  Attente... (2/20)
✅ Serveur UP — status 200  (http://localhost:8000)


In [ ]:
# ============================================================
# CELLULE 2 — Auth : register + login
# ============================================================
import requests as req, json

TEST_EMAIL    = "test@genomeer.local"
TEST_PASSWORD = "TestPass123!"
TEST_NAME     = "Test User"

# --- Register (ignore si deja existant) ---
r = req.post(f"{BASE_URL}/api/auth/register", json={
    "email": TEST_EMAIL, "name": TEST_NAME, "password": TEST_PASSWORD
})
if r.status_code == 200:
    print(f"OK Register  --> user_id={r.json()['id']}")
elif r.status_code == 400 and "already" in r.text:
    print("  Utilisateur deja existant -- OK")
else:
    print(f"ECHEC Register : {r.status_code} -- {r.text[:300]}")
    raise AssertionError(f"Register failed: {r.status_code}")

# --- Login ---
r = req.post(f"{BASE_URL}/api/auth/login", data={
    "username": TEST_EMAIL, "password": TEST_PASSWORD
})
assert r.status_code == 200, f"Login echoue : {r.status_code} -- {r.text[:300]}"
TOKEN = r.json()["access_token"]
AUTH  = {"Authorization": f"Bearer {TOKEN}"}
print(f"OK Login  --> token={TOKEN[:20]}...")

# --- Verifier /me ---
me = req.get(f"{BASE_URL}/api/auth/me", headers=AUTH)
assert me.status_code == 200, f"/me echoue : {me.text}"
print(f"OK /me  --> {me.json()}")


  Utilisateur deja existant -- OK
OK Login  --> token=eyJhbGciOiJIUzI1NiIs...
OK /me  --> {'id': 1, 'email': 'test@genomeer.local', 'name': 'Test User'}


In [ ]:
# ============================================================
# CELLULE 3 — Configurer le provider (Ollama local llama3:8b)
# ============================================================

OLLAMA_MODEL    = "llama3:8b"
OLLAMA_SOURCE   = "Ollama"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

print(f"Provider : {OLLAMA_SOURCE} @ {OLLAMA_BASE_URL}")
print(f"Modele   : {OLLAMA_MODEL}")

import json as _json
provider_payload = {
    "source":        OLLAMA_SOURCE,
    "base_url":      OLLAMA_BASE_URL,
    "api_key":       "",
    "default_model": OLLAMA_MODEL,   # champ obligatoire
}
r = req.post(f"{BASE_URL}/api/settings/provider", json=provider_payload, headers=AUTH)
if r.status_code == 200:
    print(f"OK  Provider configure : {OLLAMA_SOURCE} @ {OLLAMA_BASE_URL}")
else:
    print(f"ECHEC Provider config : {r.status_code} -- {r.text}")

model_name = OLLAMA_MODEL
r = req.post(f"{BASE_URL}/api/settings/models", json={
    "name":     model_name,
    "source":   OLLAMA_SOURCE,
    "base_url": OLLAMA_BASE_URL,
    "api_key":  "",
}, headers=AUTH)
if r.status_code == 200:
    print(f"OK  Modele ajoute : {model_name}")
elif "already" in r.text.lower() or r.status_code == 400:
    print(f"  Modele deja present : {model_name}")
else:
    print(f"ECHEC Modele : {r.status_code} -- {r.text}")

# user_models (pas 'models')
resp = req.get(f"{BASE_URL}/api/settings/models", headers=AUTH).json()
print(f"\nModeles en DB : {[m['name'] for m in resp.get('user_models', [])]}")
print(f"System default : {resp.get('system_default', {}).get('model')}")


Provider : Ollama @ http://localhost:11434/v1
Modele   : llama3:8b
OK  Provider configure : Ollama @ http://localhost:11434/v1
  Modele deja present : llama3:8b

Modeles en DB : ['llama3:8b']
System default : llama3:8b


In [ ]:
# ============================================================
# CELLULE 4 — Creer une session de chat
# ============================================================

model_name = "llama3:8b"   # repete ici pour eviter dependance sur CELLULE 3

r = req.post(f"{BASE_URL}/api/sessions", json={
    "model":            model_name,
    "interaction_mode": "auto",
}, headers=AUTH)
assert r.status_code == 200, f"Session create failed : {r.status_code} -- {r.text}"

session    = r.json()
SESSION_ID = session["id"]
print(f"OK Session creee  --> id={SESSION_ID}  modele={session.get('model')}")

sessions = req.get(f"{BASE_URL}/api/sessions", headers=AUTH).json()
print(f"Sessions actives : {[s['id'] for s in sessions]}")


OK Session creee  --> id=3  modele=llama3:8b
Sessions actives : [3, 1]


In [ ]:
# ============================================================
# CELLULE 5 — Test connexion : envoyer un message a l'agent
# ============================================================
import json

# Prompt simple adapte a llama3.1:8b (modele leger)
TEST_PROMPT = (
    "List the 3 most common bioinformatics file formats (FASTQ, FASTA, BAM). "
    "For each, write one sentence describing what it stores. "
    "Do NOT install anything. Do NOT run any code."
)

print(f"Envoi du message au session {SESSION_ID}...")
print(f"   Prompt : {TEST_PROMPT}\n")

with req.post(
    f"{BASE_URL}/api/sessions/{SESSION_ID}/messages",
    json={"message": TEST_PROMPT, "stream": True},
    headers=AUTH,
    stream=True,
    timeout=180,
) as resp:
    assert resp.status_code == 200, f"Chat failed : {resp.status_code} -- {resp.text[:300]}"

    events_received = []
    full_text       = []
    errors          = []

    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        try:
            evt = json.loads(raw_line)
            events_received.append(evt.get("type", "?"))

            if evt.get("type") == "error":
                errors.append(evt.get("text", "unknown error"))
            elif evt.get("type") in ("message", "block"):
                text = evt.get("text", "")
                if text:
                    full_text.append(text)
            elif evt.get("type") == "done":
                print("OK  Stream termine (done recu)")
        except json.JSONDecodeError:
            pass

# Rapport
print(f"\n{'='*55}")
print(f"EVENEMENTS RECUS  : {events_received}")
print(f"ERREURS           : {errors if errors else 'aucune'}")
print(f"\nREPONSE AGENT :")
print("-"*55)
print("".join(full_text)[:1500] or "(aucun texte recu)")
print("-"*55)

# Assertions
assert "done" in events_received, "ECHEC  Aucun evenement 'done' -- stream incomplet"
assert not errors,                f"ECHEC  Erreurs agent : {errors}"
assert full_text,                 "ECHEC  Aucune reponse textuelle recue"
print("\nTEST CONNEXION UI <-> AGENT : PASSED")


Envoi du message au session 3...
   Prompt : List the 3 most common bioinformatics file formats (FASTQ, FASTA, BAM). For each, write one sentence describing what it stores. Do NOT install anything. Do NOT run any code.


BioAgent_v1 CONFIGURATION
DEFAULT CONFIG :
  Path: ./data
  Run Dir: .genomeer_runs
  Timeout Seconds: 600
  Llm: ollama/llama3.1
  Temperature: 0.7
  Use Tool Retriever: True

[AGENT LLM] Constructor Override:
  LLM Model: llama3:8b
  Source: Ollama
  Base URL: http://localhost:11434/v1

================================== Ai Message ==================================

<next:QA>

FASTQ: a text-based format that stores raw, unprocessed sequencing data, including the sequence, quality scores, and optional headers.

FASTA: a plain text format that stores a sequence of nucleotides or amino acids, often used for storing genomic or protein sequences.

BAM: a binary format that stores aligned sequencing data, including the mapped sequence, alignment coordinates, and additiona


BioAgent_v1 CONFIGURATION
DEFAULT CONFIG :
  Path: ./data
  Run Dir: .genomeer_runs
  Timeout Seconds: 600
  Llm: ollama/llama3.1
  Temperature: 0.7
  Use Tool Retriever: True

[AGENT LLM] Constructor Override:
  LLM Model: llama3:8b
  Source: Ollama
  Base URL: http://localhost:11434/v1

================================== Ai Message ==================================

A simple Q&A!
================================== Ai Message ==================================

A warm greeting! 

According to our recent history, I see you said "hi" earlier. In that case, I'll respond with a friendly "Hi back!"
================================== Ai Message ==================================

<next:QA>

Note: Since this question is a simple definition/explanation, I'm routing it to QA.
================================== Ai Message ==================================

I'm happy to help!

Based on your question, I'll define metagenomics in two sentences, including AI:

Metagenomics is the study of the ge

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Vérifier que l'historique est persisté en DB   ║
# ╚══════════════════════════════════════════════════════════════╝

r = req.get(f"{BASE_URL}/api/sessions/{SESSION_ID}/messages", headers=AUTH)
assert r.status_code == 200, f"History failed : {r.status_code}"

msgs = r.json()
print(f"Messages en DB : {len(msgs)}")
for m in msgs:
    role    = m.get("role", "?")
    content = (m.get("content") or "")[:120]
    logs_n  = len(m.get("logs", []))
    print(f"  [{role.upper():10s}] {content!r}  ({logs_n} log blocks)")

assert any(m["role"] == "user"      for m in msgs), "❌ Message utilisateur absent de la DB"
assert any(m["role"] == "assistant" for m in msgs), "❌ Réponse agent absente de la DB"
print("\n✅ PERSISTANCE DB : PASSED")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Test upload (path traversal bloqué + taille)   ║
# ╚══════════════════════════════════════════════════════════════╝
import io

# 7a — Upload normal
fake_fastq = b"@SEQ_1\nACGT\n+\nIIII\n@SEQ_2\nGGCC\n+\nHHHH\n"
r = req.post(f"{BASE_URL}/api/upload",
    files={"file": ("test_sample.fastq", io.BytesIO(fake_fastq), "text/plain")},
    headers=AUTH)
assert r.status_code == 200, f"Upload échoué : {r.status_code} — {r.text}"
server_path = r.json()["path"]
print(f"✅ Upload OK → {server_path}")
assert "test_sample.fastq" in server_path, "Nom de fichier incorrect"

# 7b — Path traversal bloqué (C-1)
r = req.post(f"{BASE_URL}/api/upload",
    files={"file": ("../../etc/passwd", io.BytesIO(b"malicious"), "text/plain")},
    headers=AUTH)
if r.status_code == 400:
    print(f"✅ Path traversal bloqué (400) : {r.json()['detail']}")
else:
    # Le basename() aura tronqué au nom seul, donc HTTP 200 avec nom "passwd"
    print(f"ℹ️  Path traversal neutralisé — fichier sauvé sous : {r.json().get('path', '?')}")
    assert "etc" not in r.json().get("path",""), "❌ Path traversal NON bloqué !"

# 7c — Fichier vide (cas limite)
r = req.post(f"{BASE_URL}/api/upload",
    files={"file": ("empty.fq", io.BytesIO(b""), "text/plain")},
    headers=AUTH)
print(f"ℹ️  Fichier vide → HTTP {r.status_code}")

print("\n✅ TEST UPLOAD : PASSED")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Résumé du test de connectivité UI ↔ Agent      ║
# ╚══════════════════════════════════════════════════════════════╝

checks = [
    ("Serveur FastAPI démarré",         True),
    ("Register / Login JWT",            True),
    ("Provider Ollama configuré",       True),
    ("Session créée",                   SESSION_ID is not None),
    ("Message envoyé + stream reçu",    "done" in events_received),
    ("Erreurs agent",                   not errors),
    ("Réponse textuelle non vide",      bool(full_text)),
    ("Historique persisté en DB",       True),
    ("Upload sécurisé",                 True),
]

print(f"\n{'='*55}")
print("  RAPPORT TEST CONNECTIVITÉ UI ↔ AGENT")
print(f"{'='*55}")
for label, ok in checks:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label}")

n_ok   = sum(1 for _, ok in checks if ok)
n_fail = len(checks) - n_ok
print(f"\n{'='*55}")
print(f"  {n_ok}/{len(checks)} checks passés" + ("  — TOUT OK" if n_fail == 0 else f"  — {n_fail} ÉCHEC(S)"))
print(f"{'='*55}")

if n_fail == 0:
    print("\n🎉 UI et agent sont bien connectés et fonctionnels.")
else:
    print("\n⚠️  Vérifier les erreurs ci-dessus.")
